# Hyperparameter Tuning

## Objective

The objective of this notebook is to improve the performance of the machine learning model by finding the optimal combination of hyperparameters. GridSearchCV is used to evaluate different parameter combinations using cross-validation


In [4]:
# Import GridSearchCV library
from sklearn.model_selection import GridSearchCV
 
#Import Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

import pandas as pd


In [5]:
import sklearn
import pandas
import numpy
import joblib

print("scikit-learn:", sklearn.__version__)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)

scikit-learn: 1.4.2
pandas: 3.0.5
numpy: 2.5.2


In [6]:
# Load encoded dataset
df= pd.read_csv("../data/processed/encoded_telco_churn.csv")
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,TenureGroup_0-12,TenureGroup_13-24,TenureGroup_25-48,TenureGroup_49-72,MonthlyChargeCategory_High,MonthlyChargeCategory_Low,MonthlyChargeCategory_Medium
0,0,0,1,0,1,0,1,29.85,29.85,0,...,0,1,0,1,0,0,0,0,1,0
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,0,0,1,0,0,1,0,0,0,1
2,1,0,0,0,2,1,1,53.85,108.15,1,...,0,0,1,1,0,0,0,0,0,1
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,0,0,0,0,0,1,0,0,0,1
4,0,0,0,0,2,1,1,70.70,151.65,1,...,0,1,0,1,0,0,0,1,0,0


### Observation

The encoded dataset was successfully loaded and is ready for hyperparameter tuning.

### Prepare Features and Target Variable



In [7]:
# Seperate input features and target variable

x= df.drop("Churn", axis=1)
y= df["Churn"]


In [8]:
# Split data into training and testing sets

from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,random_state=42,test_size=0.25,stratify=y)

### Observation

The dataset was divided into training and testing sets. Hyperparameter tuning will be performed using only the training data.

### Define Hyperparameter Grid

Different values are tested for important Random Forest parameters to determine the combination that produces the best model.

In [9]:
# Define parameter combinations

param_grid ={
    "n_estimators": [100,200,300],
    "max_depth": [None,10,20],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4]
}

param_grid

{'n_estimators': [100, 200, 300],
 'max_depth': [None, 10, 20],
 'min_samples_split': [2, 5, 10],
 'min_samples_leaf': [1, 2, 4]}

###  Hyperparameter Optimization

GridSearchCV evaluates every parameter combination using cross-validation and selects the model with the best average performance.

In [10]:
# Create Random Forest model

rf= RandomForestClassifier(random_state=42)


In [11]:
# Perform GridSearchCV

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring= "accuracy",
    n_jobs=-1
)

grid_search.fit(x_train,y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 10, 20],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             scoring='accuracy')

In [14]:
# Display best paramter
grid_search.best_params_

{'max_depth': 10,
 'min_samples_leaf': 2,
 'min_samples_split': 5,
 'n_estimators': 300}

In [15]:
#Display best cross-validation score
grid_search.best_score_

0.8043213396044857

### Observation

GridSearchCV identified the combination of hyperparameters that achieved the highest croos-validation accuracy.

In [12]:
# Get the best model

best_model= grid_search.best_estimator_

In [18]:
# Predict on testing data

predictions = best_model.predict(x_test)

In [22]:
from sklearn.metrics import ( accuracy_score, classification_report,confusion_matrix)

# Display accuracy

accuracy = accuracy_score(y_test,predictions)
print(f"Accuracy: {accuracy:.2%}") 

Accuracy: 79.35%


In [23]:
# Display classification report

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1291
           1       0.64      0.50      0.56       467

    accuracy                           0.79      1758
   macro avg       0.74      0.70      0.71      1758
weighted avg       0.78      0.79      0.78      1758



In [24]:
# Display confusion metrix

confusion_matrix(y_test,predictions)

array([[1162,  129],
       [ 234,  233]], dtype=int64)

### Observation

The tuned Random Forest Classifier model was evaluated using the testing dataset. The tuned model achieved performance comparable to the default model.

### Save Tuned Model

In [13]:
import joblib

# Save the tuned model

joblib.dump(
    best_model,
    "../models/tuned_random_forest.pkl"
)
print("Tuned model saved successfully.")

Tuned model saved successfully.


### Observation

The optimized Random Forest model was saved and can be reused for future predictions without retraining.

# Summary

## Key findings

- Hyperparameter tuning was performed using GridSearchCV.
- Multiple parameter combinations were evaluated using cross-validation.
- The best-performing Random Forest model was identified.
- The tuned model was evaluated on the testing dataset.
- The optimized model was saved for deployment.